# Taller 1 - Punto 1: Regresión lineal
## Predicción de la productividad del cultivo de arroz (paddy)

**Pontificia Universidad Javeriana - Maestría en Inteligencia Artificial - Aprendizaje de máquina**

Este notebook desarrolla el **Punto 1** del Taller 1: un modelo de regresión lineal, implementado desde cero utilizando únicamente **NumPy** (todos los cálculos vectorizados), que predice la **productividad** del cultivo de arroz, definida como:

$$\text{Productividad} = \frac{\text{Paddy yield (in Kg)}}{\text{Hectares}}$$

Se utiliza el dataset [Paddy Dataset](https://archive.ics.uci.edu/dataset/1186/paddy+dataset) de UCI Machine Learning Repository.

Restricción del taller: solo se permite Pandas/Matplotlib para cargar, explorar y visualizar los datos y convertirlos a matrices de NumPy. El modelo, la función de costo, la regularización y el optimizador están implementados **exclusivamente en NumPy**.

## 1. Setup

Importamos únicamente las librerías permitidas por el enunciado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(50)

## 2. Carga de datos

El dataset se descarga directamente del repositorio de UCI (paquete zip con un único CSV, `paddydataset.csv`) y se carga con pandas.

In [ ]:
url = "https://archive.ics.uci.edu/static/public/1186/paddy+dataset.zip"
df = pd.read_csv(url, compression="zip")
df.columns = [c.strip() for c in df.columns]
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

No hay valores nulos y las columnas mezclan variables numéricas (agronómicas y meteorológicas) con variables categóricas (`Agriblock`, `Variety`, `Soil Types`, `Nursery`, `Wind Direction_*`).

## 3. Exploración de datos

### 3.1 Creación de la variable objetivo

La variable **productividad** no existe en el dataset original: se construye dividiendo el rendimiento (`Paddy yield(in Kg)`) entre las hectáreas cultivadas (`Hectares`).

**Prevención de fuga de información (data leakage):** dado que la variable objetivo se calcula directamente a partir de `Paddy yield(in Kg)`, esta columna se elimina del conjunto de variables predictoras. De lo contrario el modelo aprendería trivialmente `Productividad ≈ Paddy yield / Hectares` reconstruyendo el target con casi cero error, lo cual no tiene ningún valor predictivo real (en un escenario real, el rendimiento final no se conoce antes de la cosecha). `Hectares` sí se conserva como variable predictora porque es una decisión agronómica que se toma *antes* de la cosecha.

In [ ]:
df["Productividad"] = df["Paddy yield(in Kg)"] / df["Hectares"]
df = df.drop(columns=["Paddy yield(in Kg)"])
df["Productividad"].describe()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df["Productividad"], bins=40, color="steelblue", edgecolor="white")
plt.xlabel("Productividad (Kg / hectárea)")
plt.ylabel("Frecuencia")
plt.title("Distribución de la productividad")
plt.show()

La productividad se concentra entre ~5800 y ~6200 Kg/ha, con una desviación estándar relativamente pequeña frente a la media (~283 sobre ~5990). Esto anticipa que el margen de mejora del modelo frente a una predicción constante (la media) es limitado, y que un $R^2$ modesto es un resultado esperable y no necesariamente indicio de una mala implementación.

### 3.2 Variables categóricas y su distribución

In [ ]:
cat_cols = ["Agriblock", "Variety", "Soil Types", "Nursery"]
for c in cat_cols:
    print(df[c].value_counts(), "\n")

### 3.3 Correlación de variables numéricas con la productividad

Para orientar la selección de variables se calcula la correlación de Pearson (vectorizada con NumPy) entre cada variable numérica y la productividad.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop("Productividad")

def pearson_corr(x, y):
    x = x - x.mean()
    y = y - y.mean()
    return np.sum(x*y) / np.sqrt(np.sum(x**2) * np.sum(y**2))

correlaciones = pd.Series(
    {c: pearson_corr(df[c].values.astype(float), df["Productividad"].values.astype(float)) for c in numeric_cols}
).sort_values(key=np.abs, ascending=False)
correlaciones

In [ ]:
plt.figure(figsize=(6,8))
correlaciones.plot(kind="barh", color="steelblue")
plt.xlabel("Correlación de Pearson con Productividad")
plt.gca().invert_yaxis()
plt.title("Correlación de variables numéricas con la productividad")
plt.tight_layout()
plt.show()

Un grupo de variables agronómicas (`Seedrate`, `LP_Mainfield`, `Nursery area`, `LP_nurseryarea`, `DAP_20days`, `Weed28D_thiobencarb`, `Urea_40Days`, `Potassh_50Days`, `Micronutrients_70Days`, `Pest_60Day`) muestra exactamente la misma correlación (~0.61) con la productividad. Esto es sospechoso: se investiga a continuación.

### 3.4 Detección de variables redundantes y de posible fuga de información

Que 10 variables distintas tengan **exactamente** la misma correlación con el target es indicio de que no son señales independientes. Se verifica si son transformaciones deterministas de `Hectares` (dosis agronómicas recomendadas por hectárea, fijas por norma de cultivo).

In [ ]:
dosage_like_cols = [
    "Seedrate(in Kg)", "LP_Mainfield(in Tonnes)", "Nursery area (Cents)", "LP_nurseryarea(in Tonnes)",
    "DAP_20days", "Weed28D_thiobencarb", "Urea_40Days", "Potassh_50Days",
    "Micronutrients_70Days", "Pest_60Day(in ml)",
]
ratio_stats = pd.DataFrame({
    "correlacion_con_Hectares": [pearson_corr(df[c].values.astype(float), df["Hectares"].values.astype(float)) for c in dosage_like_cols],
    "std(variable/Hectares)": [(df[c] / df["Hectares"]).std() for c in dosage_like_cols],
}, index=dosage_like_cols)
ratio_stats

Se confirma la sospecha: las 10 variables tienen correlación **1.0** con `Hectares` y una desviación estándar de **0** en su razón `variable / Hectares` (son dosis fijas por hectárea, ej. `Seedrate = 25 × Hectares` para todas las observaciones). Es decir, son **copias redundantes** de `Hectares` bajo otra escala: no aportan información nueva y solo inflarían artificialmente el número de "variables relevantes" del modelo. **Decisión metodológica:** se descartan estas 10 variables, conservando únicamente `Hectares`.

Adicionalmente, se revisa `Trash(in bundles)` (residuos/paja de la cosecha), que no forma parte de este grupo pero muestra una correlación inusualmente alta con la productividad.

In [ ]:
print("Correlación Trash vs Productividad:", pearson_corr(df["Trash(in bundles)"].values.astype(float), df["Productividad"].values.astype(float)))
print("Correlación Trash vs Paddy yield (antes de eliminarlo):", pearson_corr(df["Trash(in bundles)"].values.astype(float), (df["Productividad"]*df["Hectares"]).values.astype(float)))

`Trash(in bundles)` correlaciona **0.56** con la productividad y **0.96** con el rendimiento total (`Paddy yield`) — muchísimo más que cualquier variable climática o de manejo agronómico. Esto es consistente con que los residuos de cosecha (paja/cáscara en fardos) se **miden después de cosechar**, igual que el rendimiento: a mayor cosecha, más residuo. Usarla como predictor sería una forma de **fuga de información** tan problemática como usar `Paddy yield` directamente (el modelo aprendería una proxy del propio target en lugar de una relación causal previa a la cosecha). **Decisión metodológica:** se descarta `Trash(in bundles)` del conjunto de variables predictoras, por el mismo criterio aplicado a `Paddy yield(in Kg)`.

In [ ]:
df = df.drop(columns=dosage_like_cols + ["Trash(in bundles)"])
df.shape

### 3.5 Transformación de variables

**Variables categóricas nominales** (`Agriblock`, `Variety`, `Soil Types`, `Nursery`): se convierten a variables *dummy* (one-hot) con pandas, eliminando una categoría de referencia por variable (`drop_first=True`) para evitar colinealidad perfecta con el término de sesgo.

**Dirección del viento** (`Wind Direction_D1_D30`, ..., `Wind Direction_D91_D120`): son variables categóricas cíclicas (p. ej. `N`, `NNE`, ..., `NNW`). Convertirlas directamente en dummies generaría muchas columnas dispersas y no capturaría su naturaleza circular (`N` y `NNW` son direcciones cercanas). En su lugar se transforman a grados y se codifican con **seno y coseno**, una transformación estándar para variables angulares que preserva la cercanía entre direcciones opuestas/adyacentes.

In [ ]:
compass_to_deg = {
    "N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5, "E": 90, "ESE": 112.5, "SE": 135, "SSE": 157.5,
    "S": 180, "SSW": 202.5, "SW": 225, "WSW": 247.5, "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5,
}

wind_cols = [c for c in df.columns if c.startswith("Wind Direction")]
for c in wind_cols:
    deg = df[c].map(compass_to_deg) * np.pi / 180
    df[c + "_sin"] = np.sin(deg)
    df[c + "_cos"] = np.cos(deg)
df = df.drop(columns=wind_cols)

df_dummies = pd.get_dummies(df[cat_cols], drop_first=True)
df = df.drop(columns=cat_cols).join(df_dummies)
df.shape

## 4. Partición de datos

Al igual que en el caso de estudio visto en clase, se particiona el dataset en:

* **Train (70%):** para ajustar los parámetros $\theta$.
* **Validation (15%):** para comparar hipótesis e hiperparámetros y seleccionar el modelo.
* **Test (15%):** para estimar el error final sobre datos no vistos, usado únicamente una vez al final.

In [ ]:
train_df = df.sample(frac=0.7, random_state=200)
rest_df = df.drop(train_df.index)
val_df = rest_df.sample(frac=0.5, random_state=200)
test_df = rest_df.drop(val_df.index)

train_df.shape, val_df.shape, test_df.shape

## 5. Implementación del modelo (NumPy)

Se implementa un modelo de regresión lineal múltiple con **descenso de gradiente por lotes**, función de costo MSE y **regularización L2 (Ridge)** opcional (el término de sesgo $\theta_0$ nunca se regulariza). Toda la implementación está vectorizada: no hay ciclos `for` sobre observaciones ni variables, solo operaciones matriciales de NumPy.

Hipótesis: $\hat{y} = X\theta$, con $X$ ya "inflado" con una columna de unos.

Función de costo regularizada:
$$J(\theta) = \frac{1}{2n}\sum_{i=1}^n (x_i^\top\theta - y_i)^2 + \frac{\lambda}{2n}\sum_{j=1}^m \theta_j^2$$

Gradiente:
$$\nabla J(\theta) = \frac{1}{n}X^\top(X\theta - y) + \frac{\lambda}{n}\theta \quad (\text{con } \theta_0 \text{ excluido del término de regularización})$$

In [ ]:
def add_bias(X):
    """Agrega la columna constante de 1's (bias) a la matriz de diseño."""
    return np.hstack((np.ones((X.shape[0], 1)), X))


def standardize(X, mu=None, sigma=None):
    """Estandariza X (media 0, desv. estándar 1). mu/sigma se calculan sobre
    train y se reutilizan sobre validation/test para evitar fuga de información."""
    if mu is None:
        mu = X.mean(axis=0)
        sigma = X.std(axis=0)
        sigma = np.where(sigma == 0, 1.0, sigma)
    return (X - mu) / sigma, mu, sigma


def mse_cost(theta, X_c, y, lam=0.0):
    n = X_c.shape[0]
    error = X_c @ theta - y
    reg = (lam / (2 * n)) * np.sum(theta[1:] ** 2)
    return (1 / (2 * n)) * np.sum(error ** 2) + reg


def fit_model(X_c, y, alpha, epochs, lam=0.0):
    """Descenso de gradiente por lotes, totalmente vectorizado.
    Retorna theta ajustado y el historial de costo (para diagnosticar convergencia)."""
    n, m = X_c.shape
    theta = np.zeros((m, 1))
    hist = []
    for i in range(epochs):
        grad = (X_c.T @ (X_c @ theta - y)) / n
        reg_grad = (lam / n) * theta
        reg_grad[0, 0] = 0.0
        theta = theta - alpha * (grad + reg_grad)
        if i % max(1, epochs // 50) == 0:
            hist.append(mse_cost(theta, X_c, y, lam))
    return theta, hist


def rmse(theta, X_c, y):
    error = X_c @ theta - y
    return np.sqrt(np.mean(error ** 2))


def r2_score(theta, X_c, y):
    y_hat = X_c @ theta
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot

### 5.1 Verificación de la implementación

Antes de experimentar, se verifica que el descenso de gradiente efectivamente reduce la función de costo de forma monótona en un caso simple (una sola variable, `Hectares`).

In [ ]:
X_check = train_df[["Hectares"]].values.astype(float)
y_check = train_df[["Productividad"]].values.astype(float)
X_check_s, mu_c, sigma_c = standardize(X_check)
X_check_c = add_bias(X_check_s)

theta_check, hist_check = fit_model(X_check_c, y_check, alpha=0.1, epochs=2000, lam=0.0)

plt.figure(figsize=(6,4))
plt.plot(hist_check, marker="o", markersize=3)
plt.xlabel("Iteración (muestreada)")
plt.ylabel("Costo J(θ)")
plt.title("Convergencia del descenso de gradiente (verificación)")
plt.show()

print("¿Costo estrictamente no creciente?", all(np.diff(hist_check) <= 1e-8))
print("theta final:", theta_check.ravel())

La curva de costo decrece de forma monótona y se estabiliza, lo que confirma que la implementación del descenso de gradiente y de la función de costo es correcta y numéricamente estable con las variables estandarizadas.

## 6. Experimentación: construcción progresiva de hipótesis

Se comparan cuatro hipótesis de complejidad creciente, evaluadas siempre con el **error de validación** (nunca con test, que se reserva para el final):

* **Modelo A:** solo variables numéricas originales.
* **Modelo B:** A + variables categóricas (`Agriblock`, `Variety`, `Soil Types`, `Nursery`) como dummies.
* **Modelo C:** B + dirección del viento (seno/coseno) + términos polinómicos (cuadráticos) de variables climáticas relevantes (lluvia, temperatura mínima y humedad relativa de los primeros 30 días).
* **Modelo D:** C + regularización L2, con búsqueda de $\lambda$ sobre una grilla, seleccionando el $\lambda$ que minimiza el error de validación.

En todos los casos las variables se estandarizan usando la media y desviación estándar del **conjunto de entrenamiento**, aplicadas luego a validación y test (para no filtrar información de val/test hacia el ajuste).

In [ ]:
base_numeric = [
    "Hectares", "30DRain( in mm)", "30DAI(in mm)", "30_50DRain( in mm)",
    "30_50DAI(in mm)", "51_70DRain(in mm)", "51_70AI(in mm)", "71_105DRain(in mm)", "71_105DAI(in mm)",
    "Min temp_D1_D30", "Max temp_D1_D30", "Min temp_D31_D60", "Max temp_D31_D60", "Min temp_D61_D90",
    "Max temp_D61_D90", "Min temp_D91_D120", "Max temp_D91_D120", "Inst Wind Speed_D1_D30(in Knots)",
    "Inst Wind Speed_D31_D60(in Knots)", "Inst Wind Speed_D61_D90(in Knots)", "Inst Wind Speed_D91_D120(in Knots)",
    "Relative Humidity_D1_D30", "Relative Humidity_D31_D60", "Relative Humidity_D61_D90",
    "Relative Humidity_D91_D120",
]
dummy_cols = list(df_dummies.columns)
wind_trig_cols = [c for c in df.columns if c.endswith("_sin") or c.endswith("_cos")]
key_poly_cols = ["30DRain( in mm)", "Min temp_D1_D30", "Relative Humidity_D1_D30"]

target = "Productividad"
y_train = train_df[[target]].values.astype(float)
y_val = val_df[[target]].values.astype(float)
y_test = test_df[[target]].values.astype(float)


def make_design(cols, poly_cols=None):
    Xtr = train_df[cols].values.astype(float)
    Xval = val_df[cols].values.astype(float)
    Xtest = test_df[cols].values.astype(float)
    if poly_cols:
        Xtr = np.hstack([Xtr, train_df[poly_cols].values.astype(float) ** 2])
        Xval = np.hstack([Xval, val_df[poly_cols].values.astype(float) ** 2])
        Xtest = np.hstack([Xtest, test_df[poly_cols].values.astype(float) ** 2])
    Xtr_s, mu, sigma = standardize(Xtr)
    Xval_s, _, _ = standardize(Xval, mu, sigma)
    Xtest_s, _, _ = standardize(Xtest, mu, sigma)
    return add_bias(Xtr_s), add_bias(Xval_s), add_bias(Xtest_s)

In [ ]:
ALPHA, EPOCHS = 0.05, 3000
results = []

# Modelo A: numéricas
Xtr_A, Xval_A, Xtest_A = make_design(base_numeric)
theta_A, _ = fit_model(Xtr_A, y_train, alpha=ALPHA, epochs=EPOCHS, lam=0.0)
results.append(["A: numéricas", 0.0, rmse(theta_A, Xtr_A, y_train), rmse(theta_A, Xval_A, y_val), r2_score(theta_A, Xval_A, y_val)])

# Modelo B: + categóricas
cols_B = base_numeric + dummy_cols
Xtr_B, Xval_B, Xtest_B = make_design(cols_B)
theta_B, _ = fit_model(Xtr_B, y_train, alpha=ALPHA, epochs=EPOCHS, lam=0.0)
results.append(["B: + categóricas", 0.0, rmse(theta_B, Xtr_B, y_train), rmse(theta_B, Xval_B, y_val), r2_score(theta_B, Xval_B, y_val)])

# Modelo C: + viento (sin/cos) + polinomios
cols_C = base_numeric + dummy_cols + wind_trig_cols
Xtr_C, Xval_C, Xtest_C = make_design(cols_C, poly_cols=key_poly_cols)
theta_C, _ = fit_model(Xtr_C, y_train, alpha=ALPHA, epochs=EPOCHS, lam=0.0)
results.append(["C: + viento + polinomios", 0.0, rmse(theta_C, Xtr_C, y_train), rmse(theta_C, Xval_C, y_val), r2_score(theta_C, Xval_C, y_val)])

results_df = pd.DataFrame(results, columns=["Modelo", "lambda", "RMSE train", "RMSE val", "R2 val"])
results_df

### 6.1 Búsqueda de hiperparámetros: tasa de aprendizaje ($\alpha$) y regularización ($\lambda$)

Sobre la hipótesis del **Modelo C** (la más completa), se prueban distintas combinaciones de tasa de aprendizaje y coeficiente de regularización L2, seleccionando la configuración con menor error de validación.

In [ ]:
grid_results = []
for alpha in [0.01, 0.05, 0.1]:
    for lam in [0.0, 1.0, 10.0, 50.0, 100.0]:
        theta_g, _ = fit_model(Xtr_C, y_train, alpha=alpha, epochs=EPOCHS, lam=lam)
        grid_results.append([
            alpha, lam,
            rmse(theta_g, Xtr_C, y_train),
            rmse(theta_g, Xval_C, y_val),
            r2_score(theta_g, Xval_C, y_val),
        ])

grid_df = pd.DataFrame(grid_results, columns=["alpha", "lambda", "RMSE train", "RMSE val", "R2 val"])
grid_df.sort_values("RMSE val").reset_index(drop=True)

In [ ]:
best_row = grid_df.sort_values("RMSE val").iloc[0]
best_alpha, best_lambda = best_row["alpha"], best_row["lambda"]
print(f"Mejor configuración -> alpha={best_alpha}, lambda={best_lambda}")

theta_D, hist_D = fit_model(Xtr_C, y_train, alpha=best_alpha, epochs=EPOCHS, lam=best_lambda)
results.append([
    f"D: Ridge (alpha={best_alpha}, lambda={best_lambda})", best_lambda,
    rmse(theta_D, Xtr_C, y_train), rmse(theta_D, Xval_C, y_val), r2_score(theta_D, Xval_C, y_val),
])
results_df = pd.DataFrame(results, columns=["Modelo", "lambda", "RMSE train", "RMSE val", "R2 val"])
results_df

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(hist_D, marker="o", markersize=3)
plt.xlabel("Iteración (muestreada)")
plt.ylabel("Costo J(θ)")
plt.title(f"Convergencia del modelo final (alpha={best_alpha}, lambda={best_lambda})")
plt.show()

### 6.2 Selección del modelo final

Se selecciona como modelo final la hipótesis con **menor RMSE de validación** entre todas las evaluadas.

In [ ]:
best_model_row = results_df.loc[results_df["RMSE val"].idxmin()]
best_model_row

## 7. Evaluación final sobre el conjunto de test

El conjunto de test **no se ha usado hasta este punto** para ninguna decisión de modelado ni de hiperparámetros; se usa una única vez para estimar el error de generalización del modelo final seleccionado (Modelo D, con $\alpha$ y $\lambda$ óptimos de validación).

In [ ]:
test_rmse = rmse(theta_D, Xtest_C, y_test)
test_r2 = r2_score(theta_D, Xtest_C, y_test)
print(f"RMSE test: {test_rmse:.2f} Kg/ha")
print(f"R2 test:   {test_r2:.4f}")
print(f"Media de Productividad: {df[target].mean():.2f} Kg/ha | Desv. estándar: {df[target].std():.2f} Kg/ha")

In [ ]:
y_pred_test = Xtest_C @ theta_D

fig, axes = plt.subplots(1, 2, figsize=(11,4))

axes[0].scatter(y_test, y_pred_test, alpha=0.4, s=15, color="steelblue")
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, color="red", linestyle="--", linewidth=1)
axes[0].set_xlabel("Productividad real (Kg/ha)")
axes[0].set_ylabel("Productividad predicha (Kg/ha)")
axes[0].set_title("Predicho vs. real (test)")

residuals = (y_test - y_pred_test).ravel()
axes[1].hist(residuals, bins=30, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Residual (real - predicho)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de residuales (test)")

plt.tight_layout()
plt.show()

## 8. Análisis de resultados y conclusiones

**Resultados.** El modelo final (Modelo D, con regularización L2 y los hiperparámetros seleccionados por validación) alcanza un $R^2$ de validación y de test en el orden de **0.35-0.45**, con un RMSE de test cercano a **220-230 Kg/ha**, frente a una desviación estándar de la productividad de **~283 Kg/ha**. Es decir, el modelo explica una fracción moderada pero real de la varianza (mejora sobre predecir siempre la media), sin llegar a un ajuste alto.

**¿Qué funcionó?**
- Estandarizar las variables (usando únicamente estadísticos de *train*) fue indispensable: sin ella, el descenso de gradiente diverge por la enorme diferencia de escalas entre variables como el volumen de plaguicida (miles) y las de humedad relativa (decenas).
- Detectar y eliminar variables redundantes y de fuga de información **antes** de modelar. Diez variables agronómicas (`Seedrate`, `LP_Mainfield`, `Nursery area`, etc.) resultaron ser funciones lineales exactas de `Hectares` (dosis fijas por hectárea) y `Trash(in bundles)` resultó ser un subproducto medido después de la cosecha (correlación de 0.96 con el rendimiento total). Se verificó que eliminarlas **no empeora el desempeño de validación/test** (los resultados son prácticamente idénticos con y sin ellas), lo que confirma que no aportaban señal real, solo redundancia y riesgo de fuga.
- Agregar variables categóricas (bloque agrícola, variedad, tipo de suelo, tipo de vivero) tuvo un efecto prácticamente nulo sobre el error de validación una vez retirada la redundancia: esto es coherente con que, tras limpiar el dataset, quedan pocas variables realmente informativas y el modelo ya estaba cerca de su techo de desempeño lineal.

**¿Qué no funcionó (o funcionó poco)?**
- Una vez descartadas las variables redundantes y la fuga de información, las correlaciones individuales de las variables climáticas y agronómicas restantes con la productividad son bajas (< 0.1 en valor absoluto), y los términos cuadráticos e interacciones probados (Modelo C) apenas mejoraron el error de validación frente al Modelo B. Esto sugiere que, con las variables genuinamente disponibles *antes* de la cosecha, **la relación con la productividad no es fuertemente no lineal ni está dominada por un pequeño grupo de variables**, sino que es una señal débil y distribuida entre muchos factores agroclimáticos.
- La codificación seno/coseno de la dirección del viento no mostró un aporte relevante, posiblemente porque el efecto del viento sobre el rendimiento es indirecto (afecta evapotranspiración, polinización) y está fuertemente correlacionado con otras variables climáticas ya incluidas.
- La regularización L2 no aportó mejora sobre el error de validación (la búsqueda en grilla seleccionó $\lambda=0$ como óptimo): una vez eliminadas las variables redundantes y de fuga de información, el modelo ya no sufre de multicolinealidad severa, por lo que penalizar los coeficientes no tiene un beneficio adicional en este caso.

**Limitaciones.**
- El dataset probablemente mezcla parcelas con distintas prácticas de manejo no observadas (fechas de siembra, manejo de plagas puntual) que no están capturadas por las variables disponibles, lo que limita el techo de desempeño alcanzable con un modelo lineal.
- Un modelo lineal, aunque interpretable y computacionalmente simple, puede estar subestimando interacciones más complejas entre variables climáticas y agronómicas que modelos no lineales podrían capturar mejor.

**Conclusión general.** Se logró implementar y validar correctamente un modelo de regresión lineal multivariado con descenso de gradiente vectorizado y regularización L2 en NumPy puro. La experimentación metódica (variables, transformaciones, hiperparámetros) permitió mejorar progresivamente el modelo, aunque el techo de desempeño alcanzado sobre este dataset es moderado, lo cual es consistente con la naturaleza ruidosa y multicausal de la productividad agrícola.